# Fluxo direto no Colab (sem clone)

Este notebook roda treino e validação sem depender de `git clone` (evita erro 403 em repositório privado).

In [ ]:
# 1) Instalar dependências
!pip -q install tensorflow matplotlib

In [ ]:
# 2) Treinar modelo
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
import os
import glob
from google.colab import drive

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

pasta_base = '/content/drive/MyDrive/Tese_IA_Jussara'

busca = glob.glob(os.path.join(pasta_base, '*MASSIVE*tfrecord*'))
if not busca:
    busca = glob.glob(os.path.join(pasta_base, '*tfrecord*'))
    busca.sort(key=os.path.getmtime, reverse=True)

if not busca:
    raise FileNotFoundError(f'Nenhum TFRecord encontrado em {pasta_base}')

caminho_arquivo = busca[0]
print(f"📂 Lendo dados de: {caminho_arquivo}")

KERNEL_SIZE = 128
READ_SIZE = 129
BATCH_SIZE = 32
EPOCHS = 40
INPUT_BANDS = ['R_1', 'NIR_1', 'NDVI_1', 'R_2', 'NIR_2', 'NDVI_2']
LABEL_BAND = 'label_chip'

def parse_and_process(example_proto):
    features_dict = {band: tf.io.VarLenFeature(tf.float32) for band in INPUT_BANDS + [LABEL_BAND]}
    parsed = tf.io.parse_single_example(example_proto, features_dict)

    inputs_list = []
    for band in INPUT_BANDS:
        dense = tf.sparse.to_dense(parsed[band], default_value=0.0)
        img = tf.reshape(dense, [READ_SIZE, READ_SIZE, 1])
        img = tf.image.resize_with_crop_or_pad(img, KERNEL_SIZE, KERNEL_SIZE)
        inputs_list.append(img)

    image_stacked = tf.concat(inputs_list, axis=-1)

    dense_lbl = tf.sparse.to_dense(parsed[LABEL_BAND], default_value=0.0)
    lbl = tf.reshape(dense_lbl, [READ_SIZE, READ_SIZE, 1])
    lbl = tf.image.resize_with_crop_or_pad(lbl, KERNEL_SIZE, KERNEL_SIZE)
    return image_stacked, lbl

print('🔢 Verificando tamanho do arquivo...')
raw_dataset = tf.data.TFRecordDataset(caminho_arquivo, compression_type='GZIP')
N_REAL = sum(1 for _ in raw_dataset)
print(f'✅ Total de amostras: {N_REAL}')

N_TRAIN = int(N_REAL * 0.8)
full_dataset = tf.data.TFRecordDataset(caminho_arquivo, compression_type='GZIP').map(parse_and_process)
train_ds = full_dataset.take(N_TRAIN).cache().shuffle(max(N_TRAIN,1)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds = full_dataset.skip(N_TRAIN).cache().batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

def build_unet(input_shape):
    inputs = layers.Input(shape=input_shape)
    c1 = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(inputs); p1 = layers.MaxPooling2D()(c1)
    c2 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(p1); p2 = layers.MaxPooling2D()(c2)
    c3 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(p2); p3 = layers.MaxPooling2D()(c3)
    c4 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(p3)
    u5 = layers.Conv2DTranspose(128, (2, 2), strides=(2, 2), padding='same')(c4)
    u5 = layers.concatenate([u5, c3])
    c5 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(u5)
    u6 = layers.Conv2DTranspose(64, (2, 2), strides=(2, 2), padding='same')(c5)
    u6 = layers.concatenate([u6, c2])
    c6 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(u6)
    u7 = layers.Conv2DTranspose(32, (2, 2), strides=(2, 2), padding='same')(c6)
    u7 = layers.concatenate([u7, c1])
    c7 = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(u7)
    outputs = layers.Conv2D(1, (1, 1), activation='sigmoid')(c7)
    return models.Model(inputs=[inputs], outputs=[outputs])

model = build_unet((KERNEL_SIZE, KERNEL_SIZE, 6))
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

checkpoint_path = os.path.join(pasta_base, 'Modelo_Checkpoint.keras')
checkpoint_cb = callbacks.ModelCheckpoint(filepath=checkpoint_path, save_best_only=False, verbose=1)

print('🔥 Iniciando Retreinamento...')
model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=[checkpoint_cb])

final_path = os.path.join(pasta_base, 'Modelo_UNet_Jussara_2025_FINAL_v2.keras')
model.save(final_path)
print(f'✅ SUCESSO! Modelo final salvo em: {final_path}')

In [ ]:
# 3) Validação visual
import tensorflow as tf
import matplotlib.pyplot as plt
import os
import glob
from google.colab import drive

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

print('--- INICIANDO PROVA REAL ---')

pasta_base = '/content/drive/MyDrive/Tese_IA_Jussara'
caminho_modelo = os.path.join(pasta_base, 'Modelo_UNet_Jussara_2025_FINAL_v2.keras')

if not os.path.exists(caminho_modelo):
    raise FileNotFoundError(f'Modelo não encontrado: {caminho_modelo}')

print(f'✅ Arquivo do modelo encontrado: {caminho_modelo}')
model = tf.keras.models.load_model(caminho_modelo)
print('✅ Modelo carregado na memória com sucesso!')

busca_dados = glob.glob(os.path.join(pasta_base, '*MASSIVE*tfrecord*'))
if not busca_dados:
    raise FileNotFoundError('Nenhum TFRecord MASSIVE encontrado para validação.')

caminho_dados = busca_dados[0]

KERNEL_SIZE = 128
READ_SIZE = 129
INPUT_BANDS = ['R_1', 'NIR_1', 'NDVI_1', 'R_2', 'NIR_2', 'NDVI_2']
LABEL_BAND = 'label_chip'

def parse_fast(example_proto):
    features_dict = {band: tf.io.VarLenFeature(tf.float32) for band in INPUT_BANDS + [LABEL_BAND]}
    parsed = tf.io.parse_single_example(example_proto, features_dict)
    inputs_list = []
    for band in INPUT_BANDS:
        dense = tf.sparse.to_dense(parsed[band], default_value=0.0)
        img = tf.reshape(dense, [READ_SIZE, READ_SIZE, 1])
        img = tf.image.resize_with_crop_or_pad(img, KERNEL_SIZE, KERNEL_SIZE)
        inputs_list.append(img)
    image_stacked = tf.concat(inputs_list, axis=-1)
    dense_lbl = tf.sparse.to_dense(parsed[LABEL_BAND], default_value=0.0)
    lbl = tf.reshape(dense_lbl, [READ_SIZE, READ_SIZE, 1])
    lbl = tf.image.resize_with_crop_or_pad(lbl, KERNEL_SIZE, KERNEL_SIZE)
    return image_stacked, lbl

dataset = tf.data.TFRecordDataset(caminho_dados, compression_type='GZIP').map(parse_fast).batch(10).take(1)
print('🔮 Gerando previsões...')
imgs, labels = next(iter(dataset))
preds = model.predict(imgs)

plt.figure(figsize=(15, 12))
for i in range(5):
    plt.subplot(5, 3, i*3 + 1); plt.imshow(imgs[i][:,:,2], cmap='RdYlGn', vmin=0, vmax=0.8); plt.axis('off')
    plt.subplot(5, 3, i*3 + 2); plt.imshow(labels[i][:,:,0], cmap='binary_r'); plt.axis('off')
    plt.subplot(5, 3, i*3 + 3); plt.imshow(preds[i][:,:,0], cmap='magma', vmin=0, vmax=1); plt.axis('off')
plt.tight_layout(); plt.show()

## Opcional: usar scripts do repositório
Se quiser usar os arquivos `.py` do repo, faça clone com token e rode `python treinamento_seguro_unet.py`.
Mas o fluxo principal acima já funciona sem clone.